# 04 — The Validation Gate

**Notebook 4 of the *Developer Guide to Disciplined Trading* series.**

> Prerequisites: [`01-foundations`](./01-foundations-techtrade-and-analysis.ipynb), [`02-morning-scan`](./02-morning-scan.ipynb), [`03-single-position-deep-dive`](./03-single-position-deep-dive.ipynb). You should have a `TradePlan` in hand and know how to read its votes.

> **Soft dependency**: this notebook requires the `[validation]` extra. If you haven't installed it:
> ```
> pip install 'openbb-techtrade[validation]'
> ```
> Without it, `obb.techtrade.validate(...)` raises `TechtradeDependencyError` with a copy-pasteable hint. Most cells below will degrade cleanly with a skip notice.

---

## The question this notebook answers

> *"Notebook 03 showed me one simulated path. The trade made money on that path. **But was the rule actually a good rule, or did I just get lucky on this one historical window?**"*

This is THE most important question in disciplined trading. Notebook 03 produced a single backtest sample. **One sample is anecdote, not evidence.** A rule that scored beautifully on one path can be catastrophically overfit — meaning it found a pattern in the noise of that specific window that won't repeat.

`obb.techtrade.validate(...)` is the gate. It takes a plan, runs the underlying confluence strategy over **walk-forward (WFO) or combinatorial purged cross-validation (CPCV) folds**, and computes:

- **PBO** — Probability of Backtest Overfitting (Bailey/López de Prado). Low is good (< 0.2 = robust; ≥ 0.5 = overfit).
- **DSR** — Deflated Sharpe Ratio. High is good (> 0.95 = robust; < 0.5 = overfit).
- **OOS Sharpe** — aggregated out-of-sample Sharpe ratio across folds.

...then returns a single one-word **verdict** ∈ `{robust, fragile, overfit}` with the precedence `overfit > robust > fragile` (any overfit trigger wins; robust requires all three favorable).

## What Alex takes away

- A complete `ValidationReport` for one plan, including the verdict and the statistical evidence behind it.
- An intuition for what each verdict actually means in practice.
- A side-by-side comparison of the **same plan under WFO vs CPCV** — when they agree vs disagree, and which to trust.
- A demonstration of the soft-dep degradation when `[validation]` isn't installed.

## Wall-clock

**2-15 minutes** per `validate` call depending on the horizon and fold count. The cell that runs validate is the slowest in the entire notebook series.

## 1. Setup probe + the soft-dep check

In [ ]:
import importlib.util

BACKTEST_AVAILABLE = importlib.util.find_spec("openbb_backtest") is not None
print(f"[validation] extra installed (openbb_backtest):  {BACKTEST_AVAILABLE}")
if not BACKTEST_AVAILABLE:
    print("\nMost cells below will SKIP cleanly. Install with:")
    print("    pip install 'openbb-techtrade[validation]'")
    print("\nThis is the same skipif pattern the integration tests use.")

In [ ]:
from openbb import obb

# Pick a ticker. NVDA is liquid + well-known + usually has a tradeable signal.
SYMBOL = "NVDA"
PRESET = "trend_follow"
print(f"Studying validation for {SYMBOL} under preset='{PRESET}'")

## 2. Build a plan to validate

`validate` takes a `TradePlan`. We need one. Re-run notebook 03's `plan()` call to produce one for our ticker.

In [ ]:
plans = obb.techtrade.plan(symbols=[SYMBOL], preset=PRESET, risk=0.01).results
if not plans:
    raise RuntimeError(f"plan() returned no plans for {SYMBOL} — score below entry_threshold today.")
plan = plans[0]
rec = plan.recommendation
print(f"Plan ready: {plan.symbol}  action={rec.action}  score={plan.signal.score:+.4f}  r:r={rec.risk_reward:.2f}")
print(f"plan.validation is currently: {plan.validation!r}")

## 3. Demonstrate the soft-dep degradation

Before the long-running validate, let's confirm what happens if `[validation]` is absent. **This cell runs even when `openbb_backtest` IS installed** — the example script catches `TechtradeDependencyError` either way. When the extra is genuinely missing, the error message has a copy-pasteable install hint.

In [ ]:
from openbb_techtrade.examples import validate_a_plan
from openbb_techtrade.validation.backtest_bridge import TechtradeDependencyError

try:
    result = validate_a_plan.main(symbol=SYMBOL, method="wfo", horizon_years=1)
    if result is None:
        print("validate_a_plan.main() returned None — typical when [validation] is absent.")
    else:
        print(f"validate_a_plan.main() returned a ValidationReport with verdict={result.verdict}")
except TechtradeDependencyError as exc:
    print(f"TechtradeDependencyError caught (this is the soft-dep contract):\n  {exc}")

### What this shows

The `validate_a_plan.py` example **degrades** when the extra is missing instead of crashing. This is the same pattern the integration test `tests/integration/test_tune.py` uses to stay green on bare installs.

If you want to see the raw raise (without the catch), the cell below tries it directly:

In [ ]:
if not BACKTEST_AVAILABLE:
    # The bare-install path: validate raises immediately.
    try:
        obb.techtrade.validate(plan=plan)
    except TechtradeDependencyError as exc:
        print(f"Raised cleanly with the install hint:\n  {exc}")
else:
    print("[validation] is installed; this cell would not trip the dependency error.")
    print("To see the error path, uninstall openbb-backtest temporarily.")

## 4. Walk-Forward validation (WFO)

If `[validation]` is installed, the next cell is the **headline call**. `method="wfo"` runs [**walk-forward optimization**](https://www.investopedia.com/terms/w/walkforward.asp): it splits the price history into rolling train/test windows, re-runs the strategy on each **out-of-sample** window, and aggregates the OOS metrics.

### Why walk-forward instead of a single train/test split?

A single [train/test split](https://en.wikipedia.org/wiki/Training,_validation,_and_test_data_sets) tells you "did the rule work on ONE OOS window" — one sample of the strategy's out-of-sample distribution. That's an anecdote by the same [n=1 argument](https://www.investopedia.com/terms/s/sample.asp) that made notebook 03's single-path P&L uninformative. Walk-forward *slides the split forward through time* and produces one OOS window per fold — typically 5-15 folds on a multi-year history — turning the anecdote into a **distribution of OOS results** you can compute statistics over. This is the [**walk-forward analysis**](https://en.wikipedia.org/wiki/Walk_forward_optimization) method, formalized by Robert Pardo in *[The Evaluation and Optimization of Trading Strategies](https://www.wiley.com/en-us/The+Evaluation+and+Optimization+of+Trading+Strategies%2C+2nd+Edition-p-9780470128015)* (Wiley 2008) and now the industry-standard bar for a "we tested it" claim.

### What "out-of-sample" actually means here

At every fold, the strategy is **fitted** (indicator params, weights, thresholds) on the train window and *evaluated* on the strictly-later test window. The test window's returns are NEVER visible to any optimizer choice. This is what makes the OOS Sharpe / DSR meaningful — a metric computed on data the strategy has never seen is a **real estimate of forward performance**, subject only to non-stationarity of the underlying market. Compare with an [in-sample Sharpe](https://www.investopedia.com/terms/i/in-sample.asp) (backtest on the same data used to pick parameters), which mathematically has to be optimistic by the [**selection-bias inflation**](https://en.wikipedia.org/wiki/Selection_bias) documented in Bailey/López de Prado 2014.

### The horizon knob

- **`horizon_years=5`** spans roughly 1260 trading days. This is the sweet spot for a swing/position-timeframe rule: enough folds (~10-15) to get a stable PBO, and enough per-fold history for indicator lookbacks (50-day / 200-day SMAs need ~1 year of pre-history each).
- **`horizon_years=1`** is ~250 days, ~2-3 folds. Faster (~30s vs 3-10 min) but the resulting verdict has wider error bars — treat it as a smoke-check, not a decision-grade validation.
- Extending to `horizon_years=10` doesn't linearly improve confidence — indicator regimes shift too much for very old data to inform current behavior. PRD §15 tests all use `horizon_years=5` as the canonical setting.

### Why the strategy is even fittable to a train window

Techtrade's rule has almost no free parameters by design — indicator periods are the [**canonical values**](https://www.investopedia.com/terms/t/technicalanalysis.asp) their inventors published (RSI 14, MACD 12/26/9, Bollinger 20/2, Ichimoku 9/26/52), and family weights are hand-picked from three presets. So the "train window" work is minimal: it's mostly burning in indicator lookbacks (200-day SMA needs 200 bars of pre-history) and computing threshold statistics like the entry-score percentile. Because there are few degrees of freedom, the [**backtest overfitting risk**](https://en.wikipedia.org/wiki/Overfitting) is proportionally low — but not zero, which is why PBO exists. See López de Prado's [*Advances in Financial Machine Learning*](https://www.wiley.com/en-us/Advances+in+Financial+Machine+Learning-p-9781119482086) (Wiley 2018) Ch. 11 for the formal treatment.

Wall-clock: **~3-10 minutes** for `horizon_years=5`. Use `horizon_years=1` if you just want to feel the surface.

In [ ]:
HORIZON = 5  # change to 1 if you want this cell to finish faster

if BACKTEST_AVAILABLE:
    print(f"Running validate(method='wfo', horizon_years={HORIZON}) on {SYMBOL}... (~3-10 min for horizon=5)")
    wfo_result = obb.techtrade.validate(
        plan=plan,
        method="wfo",
        horizon_years=HORIZON,
    )
    wfo_report = wfo_result.results
    print(f"\nDone. Verdict: {wfo_report.verdict.upper()}")
else:
    wfo_report = None
    print("[validation] not installed; skipping WFO call.")

In [ ]:
if wfo_report is not None:
    print(f"--- ValidationReport for {SYMBOL} (method=wfo, horizon={HORIZON}y) ---\n")
    print(f"  Verdict:                  {wfo_report.verdict.upper()}")
    print(f"  PBO (lower = better):     {wfo_report.pbo:.4f}")
    print(f"  Deflated Sharpe (higher): {wfo_report.deflated_sharpe:.4f}")
    print(f"  OOS Sharpe:               {wfo_report.oos_metrics.sharpe:.4f}")
    print(f"  Min-backtest-length:      {wfo_report.min_backtest_length_years:.2f} years")
    print(f"  Fold count:               {len(wfo_report.folds)}")
    print(f"\nVerdict thresholds in effect (openbb-backtest defaults):")
    for k, v in wfo_report.thresholds.items():
        print(f"    {k:<14} {v}")

### How to read the verdict

The gate's precedence rule (from `openbb_backtest.validation.report`): **`overfit > robust > fragile`**.

| If you got... | What it means | What Alex does |
|---|---|---|
| **`robust`** | PBO < 0.2 AND DSR > 0.95 AND OOS Sharpe > 0. The rule survived out-of-sample with statistical confidence. | Trust the setup. Apply notebook-03 sizing and take it. |
| **`fragile`** | Not enough evidence to call the edge real, but no evidence it's fake. Middle band. | Half-size the position. Watch behavior for the first few trades. |
| **`overfit`** | PBO ≥ 0.5 OR DSR < 0.5 OR OOS Sharpe ≤ 0. At least one trigger condemned it. | **Walk away.** This rule is fitting noise. |

### The three metrics decoded

**[PBO — Probability of Backtest Overfitting](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2308659)** (Bailey, Borwein, López de Prado, Zhu 2015). PBO answers *"if I picked the best-looking strategy from my search, what's the probability it will underperform the median strategy out-of-sample?"* Constructed via [**Combinatorially Symmetric Cross-Validation (CSCV)**](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2326253): partition the returns into N submatrices, form all `C(N, N/2)` train/test splits, rank strategies on each split, ask how often the in-sample winner ends up below-median out-of-sample. PBO = 0.5 is the coin-flip baseline (no edge; equivalent to random selection). PBO < 0.2 means fewer than 1-in-5 random-search outcomes would beat the actual one on OOS — a real edge, not overfit selection. See [Investopedia: Overfitting](https://www.investopedia.com/terms/o/overfitting.asp) for the general concept and Marcos López de Prado's [*Advances in Financial Machine Learning*](https://www.wiley.com/en-us/Advances+in+Financial+Machine+Learning-p-9781119482086) Ch. 11 for the full derivation.

**[DSR — Deflated Sharpe Ratio](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551)** (Bailey & López de Prado 2014). The regular [Sharpe ratio](https://www.investopedia.com/terms/s/sharperatio.asp) has a nasty statistical property: **the maximum Sharpe across N trials grows even if every underlying strategy is random noise** (like the maximum of N standard normals — extreme-value theory). If you tested 100 strategies and cherry-picked the highest Sharpe, that Sharpe is upward-biased by the [**multiple-testing selection bias**](https://en.wikipedia.org/wiki/Multiple_comparisons_problem). DSR deflates the observed Sharpe by an estimate of that selection inflation, then re-expresses the result as the probability that the *true* Sharpe exceeds a benchmark (usually 0). **DSR > 0.95 means the strategy has a 95%+ probability of a truly positive Sharpe** after accounting for how many other candidates existed. This is the metric that separates "we tried lots of things and something worked" from "we found a real edge." See also Bailey & López de Prado's [*Pseudo-Mathematics and Financial Charlatanism*](https://www.ams.org/notices/201405/rnoti-p458.pdf) (*Notices of the AMS*, 2014) — sharp, quotable, worth reading if you're evaluating any published backtest.

**OOS Sharpe** is exactly what it sounds like: the [Sharpe ratio](https://www.investopedia.com/terms/s/sharperatio.asp) computed on the out-of-sample returns aggregated across all folds. Included as the sanity floor — even if PBO and DSR are favorable, a negative OOS Sharpe means the rule lost money on average across the folds, and no amount of statistical laundering fixes that.

### Why is the threshold so strict?

The PRD §15 ships `pbo_robust=0.2` and `dsr_robust=0.95` deliberately — they're tight. Looser thresholds let more rules pass and more rules fail in production. The strict setting front-loads the disappointment: most candidates fail. **That's the point.** The 1-in-5 rule that DOES pass under strict thresholds is the one Alex should size into.

This is [**default-deny**](https://en.wikipedia.org/wiki/Default_deny) discipline applied to backtests. The alternative — loose thresholds, most rules pass, we'll figure out which ones actually work in production — is called [**deployment**](https://www.investopedia.com/articles/trading/06/beat-the-market.asp) by amateurs and [**the reason 80-90% of retail traders lose money over 5 years**](https://www.finra.org/investors/insights/day-trading-real-cost) by regulators. The gate is unfriendly on purpose.

## 5. Look at the individual folds

The `folds` field is a list of per-fold OOS metrics. Each fold is one walk-forward window. Looking at the distribution across folds tells Alex whether the verdict is **consistent** or driven by one outlier window.

In [ ]:
import pandas as pd

if wfo_report is not None and wfo_report.folds:
    fold_rows = []
    for f in wfo_report.folds:
        fold_rows.append({
            "fold": f.fold,
            "sharpe": round(f.metrics.sharpe, 4),
            "return": round(getattr(f.metrics, 'total_return', 0.0), 4),
            "max_dd": round(getattr(f.metrics, 'max_drawdown', 0.0), 4),
        })
    fold_df = pd.DataFrame(fold_rows)
    print(fold_df.to_string(index=False))
    print(f"\nSharpe stats: mean={fold_df['sharpe'].mean():.3f}  std={fold_df['sharpe'].std():.3f}  min={fold_df['sharpe'].min():.3f}  max={fold_df['sharpe'].max():.3f}")
else:
    print("No fold detail available (run §4 first).")

### What "consistent" looks like

- **Sharpe std < 0.5 × Sharpe mean**: most folds agree. A `robust` verdict here is well-supported.
- **Sharpe std > 1.0 × Sharpe mean**: high fold-to-fold variance. Even a robust verdict here is sensitive to which window was sampled — treat as `fragile` regardless of the label.
- **One fold has Sharpe much higher than the others**: the verdict is being carried by a single lucky window. If you remove that fold mentally, would the verdict still be robust? If no, you're looking at over-influence by an outlier.

## 6. Combinatorial Purged Cross-Validation (CPCV)

WFO splits the history into one rolling sequence. **[CPCV](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2900564)** (López de Prado, 2018) instead samples *combinations* of train/test splits to reduce path-dependency. It's slower per fold but the resulting PBO is more reliable.

### Why "combinatorial" beats "rolling"

Walk-forward answers: *"how did the rule do on ONE chronological sequence of OOS windows?"* That's a single realization of an equity curve. If 2020's specific COVID drawdown happened to fall in an OOS window in a way that flattered the rule, WFO reports the flatter number without flagging it. CPCV instead evaluates the rule on **every possible ordering of train/test splits** — combinatorially exhausting the ways history could have been sampled. The resulting distribution of Sharpes is centered on the rule's real edge; the tails are the [**path-dependent variance**](https://en.wikipedia.org/wiki/Path_dependence) that WFO conflates with signal. [Investopedia: Cross-Validation](https://www.investopedia.com/terms/c/cross-validation.asp) covers the underlying concept.

### Purging + embargoing (the "purged" in CPCV)

Financial time series have a nasty property: **train and test windows can leak into each other** even when they're chronologically separated. A 21-day forward-return label computed on the last day of the train window uses prices FROM the test window. This is a subtle [**data leakage**](https://en.wikipedia.org/wiki/Leakage_(machine_learning)) that inflates apparent OOS Sharpe. CPCV's "purge" removes any train sample whose label period overlaps the test window; its "embargo" removes samples on the *inside boundary* of the test window that might be contaminated by autocorrelation of returns. This is why CPCV takes longer per fold — the eligible-training-samples set is smaller after purging. See López de Prado's [*Advances in Financial Machine Learning*](https://www.wiley.com/en-us/Advances+in+Financial+Machine+Learning-p-9781119482086) Ch. 7 for the full protocol.

### When WFO and CPCV disagree

Run the same plan under `method="cpcv"` and compare the two verdicts. Disagreement is **informative**, not a bug:

- **WFO robust, CPCV fragile** — the rule works on the chronological sequence but not on permuted ones. Usually a sign the rule benefits from a specific market regime (e.g., "worked because 2022 was a trend year") that happened to fall in the OOS windows of the WFO split.
- **WFO fragile, CPCV robust** — the rule is fine but WFO's specific window slicing happened to catch a bad OOS regime. Trust CPCV; run more folds if you want the extra confidence.
- **Both agree** — highest-confidence answer; no need to reconcile.

Default: **when they disagree, take the more conservative verdict.** Alex would rather skip a real edge than trade a fake one; the [**cost of a false positive**](https://en.wikipedia.org/wiki/Type_I_and_type_II_errors) (real money lost on a bad rule) dominates the cost of a false negative (missed opportunity, no capital lost).

Wall-clock: **~5-15 minutes** (CPCV evaluates more folds).

In [ ]:
if BACKTEST_AVAILABLE:
    print(f"Running validate(method='cpcv', horizon_years={HORIZON}) on {SYMBOL}... (~5-15 min)")
    cpcv_result = obb.techtrade.validate(
        plan=plan,
        method="cpcv",
        horizon_years=HORIZON,
    )
    cpcv_report = cpcv_result.results
    print(f"Done. CPCV verdict: {cpcv_report.verdict.upper()}")
else:
    cpcv_report = None
    print("[validation] not installed; skipping CPCV call.")

In [ ]:
if wfo_report is not None and cpcv_report is not None:
    print(f"--- WFO vs CPCV comparison for {SYMBOL} ---\n")
    print(f"  {'metric':<22} {'WFO':>12} {'CPCV':>12}")
    print(f"  {'-'*22} {'-'*12} {'-'*12}")
    print(f"  {'Verdict':<22} {wfo_report.verdict.upper():>12} {cpcv_report.verdict.upper():>12}")
    print(f"  {'PBO':<22} {wfo_report.pbo:>12.4f} {cpcv_report.pbo:>12.4f}")
    print(f"  {'Deflated Sharpe':<22} {wfo_report.deflated_sharpe:>12.4f} {cpcv_report.deflated_sharpe:>12.4f}")
    print(f"  {'OOS Sharpe':<22} {wfo_report.oos_metrics.sharpe:>12.4f} {cpcv_report.oos_metrics.sharpe:>12.4f}")
    print(f"  {'Folds':<22} {len(wfo_report.folds):>12} {len(cpcv_report.folds):>12}")

    if wfo_report.verdict == cpcv_report.verdict:
        print(f"\n  ✓ Both methods agree: {wfo_report.verdict.upper()}")
    else:
        print(f"\n  ⚠ Methods DISAGREE: WFO={wfo_report.verdict} vs CPCV={cpcv_report.verdict}")
        print("    Treat as the weaker of the two. Alex defaults to the more conservative verdict.")

## 7. The IC harness — per-vote empirical justification (bd-znw)

`validate` gives you a verdict on *the whole rule*. But under the hood the composite score is a **weighted sum of individual votes** (see notebook 03 §2 attribution). Two natural questions:

1. **Which votes actually predict returns?** A vote whose signal has zero forward-return correlation contributes noise to the composite. Reading vote-level attribution in notebook 03 shows you *what the vote said*; it does NOT tell you *whether the vote was right*.
2. **When the extended panel adds new votes, do they add signal or just complexity?** The confluence-panel expansion program (bd-tik / bd-luy / bd-40v / bd-z43 / bd-alj) is architecturally allowed to add up to 20 new votes across the four families. Each new vote has to earn its slot.

The **Information Coefficient (IC) harness** answers both. IC = [Spearman rank correlation](https://www.investopedia.com/terms/s/spearman-correlation.asp) between a vote's value and forward returns over a fixed horizon (default 21 bars). Positive IC = the vote predicts direction. Bounded to [-1, +1]; typical values in equity are `|IC| = 0.02 - 0.08` per vote.

### What the harness reports

For each vote in the extended panel, the harness produces:

| Field | Meaning |
|---|---|
| `ic` | Spearman ρ between vote value and forward return, pooled across the basket |
| `ic_std` / `ic_ir` | Cross-fold IC standard deviation and IC information ratio (`ic / ic_std`) — the [**Grinold IR**](https://en.wikipedia.org/wiki/Information_ratio) applied to the vote itself |
| `null_p` | Permutation-test p-value: probability a random-shuffle vote would produce IC this large |
| `bh_p` | Benjamini-Hochberg-corrected p-value for multiple-testing across all votes evaluated |
| `regime_ic` | IC broken down by bull / bear / high-vol / low-vol regime (per openbb-regime PR #349) |

**Ship criterion (per bead bd-w1g7):** a vote joins the ship allowlist only if net IC-IR > 0.7 AND null-p < 0.05 AND BH-p < 0.05 AND positive net IC in ≥ 2 of 4 regimes. That's why `ichimoku_cloud` is currently excluded from `SHIP_ENABLED_EXTENDED_TREND_VOTES` — the correlated pair with `ema_cross` failed the decorrelation gate before it could even be IC-scored.

### Where the harness lives

The IC harness is **not yet integrated in-notebook** — it's currently blocked on **bd-69px** (the point-in-time XLK universe builder that survives biases) and **bd-a4cl** (external openbb-regime PR #349 for regime segmentation). Once both land, this section will surface live IC / DSR delta / shadow-mode divergence stats via `openbb_techtrade.engine.panel_eval`. Until then, treat this as a signposting section — the design and the CI harness both exist; only the wall-clock evidence display is deferred.

### DSR delta — did the extension add signal?

Complementary to per-vote IC: the ensemble question. After adding a family of extended votes, does the **whole composite's DSR go up or down** vs the classic panel? DSR delta > 0 means the new votes added incremental risk-adjusted signal; DSR delta ≤ 0 means the added complexity produced no lift (or actually hurt). This is the answer notebook 04's `validate` output should carry alongside the verdict once bd-w1g7 lands.

### Shadow-mode divergence — how different are the two panels in practice?

Shadow mode (see `openbb_techtrade.engine.panel_eval.ShadowDiff`) computes BOTH the classic and extended composite for every scan and logs the delta. Over a few weeks of scans, the accumulated distribution of `score_delta` values tells you whether the extended panel meaningfully changes the ranking (large deltas = the extension is doing real work) or just adds noise around zero. This is the empirical foundation for eventually defaulting the extended panel to `True` in a future release.

Concretely, once the harness surfaces here, this section will render:

```
--- Per-vote IC (extended trend family) ---
   vote_name         ic      ic_ir    null_p    bh_p   ship?
   aroon_osc      0.042      0.89     0.021   0.041     yes
   ichimoku_cloud 0.038      0.71     0.048   0.089     no  (bd-hpxh decorrelation)

--- Composite DSR delta (extended vs classic) ---
   NVDA:  +0.12     PG: +0.04     XOM: -0.02     SPY: +0.07     PLTR: skip

--- Shadow-mode divergence (last 30 scans) ---
   |score_delta| mean:  0.038
   |score_delta| p95:   0.114
   Sign-change rate:    4.2% of ranked positions
```

See `docs/superpowers/specs/2026-07-10-ensemble-lift-validation-design.md` for the full ensemble-lift acceptance methodology.

## 8. Attaching the verdict to the plan

`validate` returns the report; the design contract (PRD §15) is that the report **attaches to `plan.validation`**. The bridge function `validate_plan` (from `openbb_techtrade.validation.backtest_bridge`) returns both the updated plan AND the report. Alex's downstream code can then `if plan.validation is not None and plan.validation.verdict == "robust":`.

Below we use the bridge directly to get the attached plan back.

In [ ]:
if BACKTEST_AVAILABLE:
    import asyncio
    from openbb_techtrade.validation.backtest_bridge import validate_plan

    # Use the already-computed WFO horizon for speed; this is the canonical
    # downstream pattern (verdict attached to the plan, ready for the rest
    # of Alex's pipeline to consume).
    updated_plan, attached_report = asyncio.run(
        validate_plan(plan, method="wfo", horizon_years=HORIZON)
    )
    print(f"updated_plan.validation is now: {attached_report.verdict}")
    print(f"  Same as plan.validation? {updated_plan.validation is attached_report}")
    print(f"  Original plan unchanged?  {plan.validation is None}")

### Why is the original `plan.validation` still None?

Immutable discipline. `validate_plan` returns a **new plan** with the report attached (`plan.model_copy(update={"validation": report})`); the original `plan` is untouched. This means Alex can validate the same plan under multiple methods without state-leak between calls.

## 9. Now what? The decision table

Alex has a verdict. He's NOT going to take a trade on a single ticker just because the verdict says robust. The verdict is **one input** to his decision. Here's the table he wrote on the wall:

In [ ]:
decision_table = pd.DataFrame([
    {"verdict": "robust",  "r:r >= 2.0": "yes", "vote agreement": "3/4 families", "action": "Full size (notebook 03 qty)"},
    {"verdict": "robust",  "r:r >= 2.0": "yes", "vote agreement": "2/4 families", "action": "Half size"},
    {"verdict": "robust",  "r:r >= 2.0": "no",  "vote agreement": "any",         "action": "SKIP — bad math regardless of verdict"},
    {"verdict": "fragile", "r:r >= 2.0": "yes", "vote agreement": "3/4 families", "action": "Quarter size, observe 5 trades"},
    {"verdict": "fragile", "r:r >= 2.0": "yes", "vote agreement": "2/4 families", "action": "SKIP — not enough evidence"},
    {"verdict": "overfit", "r:r >= 2.0": "any", "vote agreement": "any",         "action": "SKIP — fitting noise; don't trade"},
])
decision_table

### Calibrate this for yourself

This table is **Alex's** discipline; it's not the engine's. You'll want a different one. Two principles to keep:

1. **`overfit` is always a skip.** Never an exception. The whole point of running validate is to honor this column.
2. **Position sizing is the lever.** Verdict + R:R + vote agreement should produce a sizing fraction (0.0 - 1.0 × notebook-03 qty), not just a yes/no. "Don't trade" and "full size" are the endpoints of a continuum, not the only options.

## 10. What's next

- **Notebook 05 — Per-Sector Tuning**: `obb.techtrade.tune(segment, ...)`. If validate said `fragile` or `overfit` for a sector, could *better indicator periods* turn it into `robust`? [Tuneta](https://github.com/8W9aG/tuneta) proposes new periods, validate gates them, the gate persists only the robust ones to disk. Only the strategy periods that survive PBO+DSR ever influence live trading — the same discipline this notebook enforces at the rule level, applied per-parameter.
- **Notebook 06 — Audit and Replay**: load yesterday's actionable Excel + the validation verdicts that came with it, replay what Alex did vs what the engine said. The [**trading journal**](https://www.investopedia.com/articles/trading/09/trading-journal.asp) is the single highest-leverage skill-building habit retail traders almost never adopt. Notebook 06 makes it a data pipeline, not a discretion-eroding notebook page.

## Recommended reading on the math

**The foundational papers (mandatory if you're serious):**

- **PBO** — Bailey, Borwein, López de Prado & Zhu, [*The Probability of Backtest Overfitting*](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2326253) (*Journal of Computational Finance*, 2016). Formalizes CSCV and gives you the intuition for why "look at more strategies" catastrophically inflates apparent Sharpe.
- **DSR** — Bailey & López de Prado, [*The Deflated Sharpe Ratio*](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551) (*Journal of Portfolio Management*, 2014). The multiple-testing correction that turns a raw Sharpe into a probability.
- **Both, packaged with the rhetorical hammer** — Bailey & López de Prado, [*Pseudo-Mathematics and Financial Charlatanism*](https://www.ams.org/notices/201405/rnoti-p458.pdf) (*Notices of the AMS*, 2014). Short. Angry. Explains why 90% of published backtests are noise. **Read this before evaluating any strategy pitch, ever.**

**The applied bridge:**

- **CPCV & the modern ML approach to trading** — Marcos López de Prado, *[Advances in Financial Machine Learning](https://www.wiley.com/en-us/Advances+in+Financial+Machine+Learning-p-9781119482086)* (Wiley, 2018), Ch. 7 (Cross-Validation in Finance) + Ch. 11 (Backtest Statistics) + Ch. 12 (Backtesting through Cross-Validation). The textbook that made overfitting-aware backtesting mainstream in quant.
- **Walk-forward, at the practitioner level** — Robert Pardo, *[The Evaluation and Optimization of Trading Strategies](https://www.wiley.com/en-us/The+Evaluation+and+Optimization+of+Trading+Strategies%2C+2nd+Edition-p-9780470128015)* (Wiley, 2008). The book that popularized walk-forward as a discipline. Older than the DSR/PBO papers but still the best explanation of why rolling-window backtests beat static ones.
- **Retail-scale examples of overfitting-induced disaster** — Ernie Chan, *[Algorithmic Trading](https://www.wiley.com/en-us/Algorithmic+Trading%3A+Winning+Strategies+and+Their+Rationale-p-9781118460146)* (Wiley, 2013). Multiple case studies of "great backtest, live-money disaster" with the specific technical reasons.

**The regulator-level warnings (short, free, sobering):**

- [FINRA on retail day-trading outcomes](https://www.finra.org/investors/insights/day-trading-real-cost) — cites the 80-90% failure rate at 5 years.
- [SEC investor.gov — Backtesting risks and disclosures](https://www.investor.gov/introduction-investing/investing-basics/how-stock-markets-work/how-market-works) — regulator-required disclosures the industry has to make about backtest validity.
- [Investopedia: Backtesting Pitfalls](https://www.investopedia.com/terms/b/backtesting.asp) — beginner-friendly catalog of the traps.

All three of these — the papers, the applied books, the regulator warnings — converge on the same lesson: **the default outcome of a "great backtest" is disappointment when live money hits it. The gate ships with strict defaults because the disappointment costs more than the missed opportunities.**

---

*End of notebook 04. Series: "A Developer Guide to Disciplined Trading". Maintained on the `trading_technicals` branch of `prajoria/OpenBB`.*